In [1]:
from all_needed_modules import *
from all_FFOPT_classes import *
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

system tools imported
basic tools imported
chemistry tools imported
ML  imported
all modules and data extracted


In [2]:
#THIS ARE ALL INFORMATIONS THAT WILL BE MOVED AND SHOULD BELONG IN A STRUCTURE_INFORMATION.PY FILE
Structure_name = 'GF334'
structure_carfile = './data/GF334_v2/GF334.car'
structure_mdffile = './data/GF334_v2/GF334.mdf'
DFT_ref_path = './data/GF334_v2/DFT_reference'
GMX_ref_path = '/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical'
charge_file = '/home/uccaset/Scratch/FFOPTI/data/GF334_v2/REPEAT_charges.resp'
groupid='MOF'
atom_types = ['Zn3', 'C_2', 'C_1', 'H_', 'N_R']  # fixed order is important
train = ['lj']
size = [26.494, 26.494, 29.203]
reference_force_field = 'D2'

In [3]:
#information extraction
Structure = About_Structure(name=Structure_name, carfile=structure_carfile, mdffile=structure_mdffile, chargefile=charge_file)
DFT_ref = DFT_reference(Structure, Structure_name, DFT_ref_path, structure_file='GF334.pdb')
GMX_ref = GMX_read_reference(Structure_name, GMX_ref_path)
initial_opti_info = ParOpti_preparation(Structure_name,Structure, DFT_ref_path, GMX_ref_path)
#Obtaining the initial GMX parameters
param = GMX_ref.ref_FF_parameters()
print('information obtained')

Obtained force field parameters for 4 unique parameter sets.
information obtained


In [4]:
#GETTING TRAJECTORIES
traj_dcd_path = f'{DFT_ref_path}/md-pos-1.dcd'
frame_list = read(traj_dcd_path, index=':') #turning each frames into ase objects
# reading indx list
with open(f"{GMX_ref_path}/ffopti_backup/numbers.json", "r") as f:
    indx = json.load(f)
frames = [frame_list[i] for i in indx]

ref = frames[0] #choosing a reference frame for NVT to set PBC and cell dimensions

for atoms in frames:
    atoms.set_cell(size)
    atoms.set_pbc(True)
    atoms.wrap()  # optional, wraps positions inside box
n_of_types = Structure.atom_types()

In [5]:
order = ['lj', 'bond', 'angle', 'dihedral']

In [6]:
flattened_parameter = []
all_indices = []

for a, l1 in enumerate(list(param)):

    # enforce ordering regardless of input
    active = [x for x in order if x in train]

    parts = []
    indices = {}
    offset = 0

    for key in active:

        if key == 'lj':
            arr = param[f'{l1}']['lj'][['sigma', 'epsilon']].to_numpy().ravel()
            parts.append(arr)

            n = len(arr)
            if 'lj_sigma' not in indices:
                indices['lj_sigma']   = list(range(offset, offset+n, 2))
                indices['lj_epsilon'] = list(range(offset+1, offset+n, 2))
            offset += n

        elif key == 'bond':
            arr = param[f'{l1}']['bonds'][['length', 'force_const']].to_numpy().ravel()
            parts.append(arr)

            n = len(arr)
            if 'bond_rij' not in indices:
                indices['bond_rij'] = list(range(offset, offset+n, 2))
                indices['bond_kij'] = list(range(offset+1, offset+n, 2))
            offset += n

        elif key == 'angle':
            arr = param[f'{l1}']['angles'][['angle', 'force_const']].to_numpy().ravel()
            parts.append(arr)

            n = len(arr)
            if 'angle_angle' not in indices:
                indices['angle_angle'] = list(range(offset, offset+n, 2))
                indices['angle_kijk'] = list(range(offset+1, offset+n, 2))
            offset += n

        elif key == 'dihedral':
            arr = param[f'{l1}']['dihedrals'][['T1', 'T2', 'T3', 'T4', 'T5', 'T6']].to_numpy().ravel()
            parts.append(arr)

            n = len(arr)
            #if 'dih_T1' not in indices:
            #indices['dih_T1'] = list(range(offset, offset+n, 2))
            #indices['dih_T2'] = list(range(offset+1, offset+n, 2))
            #offset += n

    # safely concatenate
    flatie = np.concatenate(parts) if parts else np.array([])

    flattened_parameter.append(flatie)
    all_indices.append(indices)

In [7]:
len(param)

4

In [8]:
flattened_parameter

[array([0.246155, 0.5188  , 0.343085, 0.4393  , 0.343085, 0.4393  ,
        0.257113, 0.1841  , 0.326069, 0.2887  ]),
 array([0.195998, 0.0523  , 0.385   , 0.58576 , 0.355   , 0.29288 ,
        0.25    , 0.2092  , 0.325   , 0.71128 ])]

In [9]:
feature_length = len(flattened_parameter[0])
print(feature_length)

10


In [21]:
#EXTRACTING SHAPE
lj_shape = param[reference_force_field]['lj'][['sigma','epsilon']].shape
bonds_shape = param[reference_force_field]['bonds'][['length', 'force_const']].shape
angles_shape = param[reference_force_field]['angles'][['angle', 'force_const']].shape
#dihedrals_shape = param['D3']['dihedrals'][['periodicity', 'phase', 'force_const']].shape
dihedrals_shape = param[reference_force_field]['dihedrals'][['T1', 'T2', 'T3','T4', 'T5', 'T6']].shape

In [11]:
#EXTRACTING THE LOSSES
file_path = f"{GMX_ref_path}/ffopti_backup/param_type_loss.pkl"
if os.path.exists(file_path):
    average_loss = pd.read_pickle(file_path)
else:
    average_loss = initial_opti_info.average_type_loss_per_D(metrics="MAPE")
    average_loss.to_pickle(f"{GMX_ref_path}/ffopti_backup/param_type_loss.pkl")

In [12]:
average_loss

,1,2
Zn3,1.416719,1.434663
C_2,6.847904,3.929089
C_1,2.623223,3.129213
H_,4.204055,3.568191
N_R,1.791548,2.376170


In [13]:
#If we did read from the pkl file, we now check if there are any losses that were not updated
D_vals_files = list(param.keys())
pkl_list = average_loss.columns

missing_param_loss = []
for a in D_vals_files:
    num=a[1:]
    if int(num) not in pkl_list:
        missing_param_loss.append(num)
print(f'{len(missing_param_loss)} missing from backup file')

for a in missing_param_loss:
    d_folder = f'{GMX_ref_path}/D{a}'
    single_calc_tools = single_loss_calc(Structure_name,Structure, DFT_ref_path,d_folder)
    new_loss = single_calc_tools.average_type_loss_per_D()
    average_loss = average_loss.join(new_loss)
    print(f'D{a}' added to average_loss list')

0 missing from backup file


In [15]:
bounds = []

lj_sigma = set(indices.get('lj_sigma', []))
lj_epsilon = set(indices.get('lj_epsilon', []))

bond_rij = set(indices.get('bond_rij', []))
bond_kij = set(indices.get('bond_kij', []))


angle_angle = set(indices.get('angle_angle', []))
angle_kijk = set(indices.get('angle_kijk', []))

#dih_t1 = set(indices.get('dih_T1', []))
#dih_t2 = set(indices.get('dih_T2', []))

for i, x in enumerate(initial_guess):

    # LJ rules
    if i in lj_sigma or i in bond_rij:
        bounds.append((0.90 * x, 1.10 * x))
    if i in lj_epsilon:
        bounds.append((0.75 * x, 1.25 * x))

    # bond rules
    if 'bond' in train and i in bond_rij:
        bounds.append((0.90 * x, 1.10 * x))
    if 'bond' in train and i in bond_kij:
        bounds.append((0.70 * x, 1.30 * x))

        
    # angle rules
    if 'angle' in train and i in angle_angle:
        bounds.append((0.90 * x, 1.10 * x))
    if 'angle' in train and i in angle_kijk:
        bounds.append((0.70 * x, 1.30 * x))

    # dihedral rules
    #if 'dihedral' in train and i in dih_T1:
        #bounds.append((0.70 * x, 1.30 * x))
    #if 'dihedral' in train and i in dih_T2:
        #bounds.append((0.70 * x, 1.30 * x))
        

    # fallback
    #if len(bounds) < i + 1:
        #bounds.append((0.95 * x, 1.05 * x))

In [53]:
y_train = np.array(y_train)          # convert list → numpy array
y_train = y_train.reshape(-1, 1)  

In [56]:
    from sklearn.preprocessing import StandardScaler
    from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel
# 2. Initialize the Scalers
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
     
    # 3. Scale the training data
    # It is vital to fit the scaler only on existing data and transform it
    X_scaled = scaler_X.fit_transform(X_train)
    y_scaled = scaler_y.fit_transform(y_train).flatten()
     

In [58]:
    from sklearn.preprocessing import StandardScaler
# 2. Initialize the Scalers
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
     
    # 3. Scale the training data
    # It is vital to fit the scaler only on existing data and transform it
    X_scaled = scaler_X.fit_transform(X_train)
    y_scaled = scaler_y.fit_transform(y_train)
     
    # 4. Define and fit the GP model with an improved kernel
    # Added a WhiteKernel to account for noise in the GROMACS simulations
    #kernel = C(1.0) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=1e-5)
    kernel = C(1.0, (1e-3, 1e4)) * Matern(
        length_scale=np.ones(feature_length),          # 8 features in X_train
        length_scale_bounds=(1e-2, 1e2),  # optimizer can tune each scale
        nu=2.5                            # smoothness of function
    ) + WhiteKernel(noise_level=1e-5)
                    
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10,random_state=42)
    #gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=1e-6, normalize_y=True, random_state=42)
    gp.fit(X_scaled, y_scaled)
     
    # 5. When predicting the next sample, remember to scale the input
    # and inverse-transform the output to get physical units back
    def predict_loss(new_params):
        # Ensure input is a 2D array for the scaler
        new_params_scaled = scaler_X.transform(np.atleast_2d(new_params))
        y_pred_scaled, sigma_scaled = gp.predict(new_params_scaled, return_std=True)
        # Transform back to original units
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
        # Sigma needs scaling by the standard deviation of y
        sigma = sigma_scaled * np.sqrt(scaler_y.var_)
        return y_pred.flatten(), sigma.flatten()
    
    # NOTE: normalize_y=True fixes the ConvergenceWarning by scaling target data
    
    print("Learned kernel:", gp.kernel_)


Learned kernel: 0.198**2 * Matern(length_scale=[0.0203, 0.144, 0.0441, 52.3, 17.1, 3.42, 30.6, 16.4, 0.0558, 37.2], nu=2.5) + WhiteKernel(noise_level=0.961)


#BUILDING THE GAUSSIAN PROCESS OF BO METHOD


kernel = C(1.0, (1e-3, 1e4)) * Matern(
    length_scale=np.ones(feature_length),          # 8 features in X_train
    length_scale_bounds=(1e-2, 1e2),  # optimizer can tune each scale
    nu=2.5                            # smoothness of function
)

# NOTE: normalize_y=True fixes the ConvergenceWarning by scaling target data
gp = GaussianProcessRegressor(kernel=kernel,
                              n_restarts_optimizer=10,
                              alpha=1e-6,
                              normalize_y=True,
                              random_state=42)

gp.fit(X_train, y_train)
print("Learned kernel:", gp.kernel_)

learned_length_scales = gp.kernel_.k2.length_scale

In [59]:
D_val

2

In [17]:
# Sensitivity is inversely proportional to length scale and these allows us to screen over the most relevant parameters if we want to reduce the search-set.
sensitivities = 1.0 / learned_length_scales

print(f"--- Sensitivity Analysis ---")
print("\n".join(
    f"Variable x{i} Sensitivity: {val:.4f}"
    for i, val in enumerate(sensitivities, start=1)
))

--- Sensitivity Analysis ---
Variable x1 Sensitivity: 1.2916
Variable x2 Sensitivity: 33.2371
Variable x3 Sensitivity: 0.1403
Variable x4 Sensitivity: 0.0905
Variable x5 Sensitivity: 0.5687
Variable x6 Sensitivity: 0.0824
Variable x7 Sensitivity: 1.0588
Variable x8 Sensitivity: 0.8111
Variable x9 Sensitivity: 1.9491
Variable x10 Sensitivity: 79.1268


In [19]:
ACQ_FUNC = expected_improvement
# Direction of optimisation
def min_obj(X):
    return -ACQ_FUNC(X.reshape(1, -1), gp, np.min(y_train)) #use this if just the average

In [20]:
# Optimization Loop: Select the number of BO rounds  ---
n_iterations = 100
print("\n--- Starting Optimization ---")

for i in range(n_iterations):
    y_best = np.min(y_train)

    best_x_next = None
    max_acq = -np.inf

    # Restart optimizer to avoid local traps in acquisition surface
    for _ in range(10):
        bounds = np.array(bounds)  # shape should be (n_features, 2)
        x0 = np.random.uniform(bounds[:, 0], bounds[:, 1])
        res = minimize(min_obj, x0=x0, bounds=bounds, method='L-BFGS-B')
        if -res.fun > max_acq:
            max_acq = -res.fun
            best_x_next = res.x

print(f"Iteration {i+1}: Suggested new parameters: {best_x_next}")



--- Starting Optimization ---
Iteration 100: Suggested new parameters: [0.18075917 0.05987926 0.36738301 0.58385302 0.32331992 0.26543198
 0.2681806  0.16989995 0.3206167  0.72031594]


In [22]:
idx = 0
x = best_x_next
#LJ parameter
if 'lj' in train:
    nlj = lj_shape[0] * lj_shape[1]
    lj_new = x[idx:idx + nlj].reshape(lj_shape)
    idx += nlj

    lj_df_new = param[reference_force_field]['lj'].copy()
    lj_df_new['sigma'] = lj_new[:, 0]
    lj_df_new['epsilon'] = lj_new[:, 1]

else:
    lj_df_new = param[reference_force_field]['lj'].copy()

#Bond Parameter
if 'bond' in train:
    nbond = bonds_shape[0] * bonds_shape[1]
    bonds_new = x[idx:idx + nbond].reshape(bonds_shape)
    idx += nbond
    
    bonds_df_new = param[reference_force_field]['bonds'].copy()
    bonds_df_new['length'] = bonds_new[:, 0]
    bonds_df_new['force_const'] = bonds_new[:, 1]
else:
    bonds_df_new = param[reference_force_field]['bonds'].copy()

#angle Parameter
if 'angle' in train:
    nangle = angles_shape[0] * angles_shape[1]
    angles_new = x[idx:idx + nangle].reshape(angles_shape)
    idx += nangle
    
    angles_df_new = param[reference_force_field]['angles'].copy()
    angles_df_new['angle'] = angles_new[:, 0]
    angles_df_new['force_const'] = angles_new[:, 1]
else:
    angles_df_new = param[reference_force_field]['angles'].copy()


#angle Parameter
if 'dihedral' in train and dihedrals_shape[1] == 3:
    ndihedral = dihedrals_shape[0] * dihedrals_shape[1]
    dihedrals_new = x[idx:idx + ndihedral].reshape(dihedrals_shape)
    idx += ndihedral
    
    dihedrals_df_new = param[reference_force_field]['dihedrals'].copy()
    dihedrals_df_new['periodicity'] = dihedrals_new[:, 0]
    dihedrals_df_new['phase'] = dihedrals_new[:, 1]
    dihedrals_df_new['force_const'] = dihedrals_new[:, 2]
elif 'dihedral' in train and dihedrals_shape[1] == 6:
    ndihedral = dihedrals_shape[0] * dihedrals_shape[1]
    dihedrals_new = x[idx:idx + ndihedral].reshape(dihedrals_shape)
    idx += ndihedral
    
    dihedrals_df_new = param[reference_force_field]['dihedrals'].copy()
    dihedrals_df_new['T1'] = dihedrals_new[:, 0]
    dihedrals_df_new['T2'] = dihedrals_new[:, 1]
    dihedrals_df_new['T3'] = dihedrals_new[:, 2]
    dihedrals_df_new['T4'] = dihedrals_new[:, 3]
    dihedrals_df_new['T5'] = dihedrals_new[:, 4]
    dihedrals_df_new['T6'] = dihedrals_new[:, 5]
else:
    dihedrals_df_new = param[reference_force_field]['dihedrals'].copy()

new_D = f'D{len(list(param))+1}'
param[new_D] = {
    'lj': lj_df_new,
    'bonds': bonds_df_new,
    'angles': angles_df_new,
    'dihedrals': dihedrals_df_new
}

print(param[new_D])


{'lj':   atom func    mass  charge ptype     sigma   epsilon
0  Zn3    1  65.380     0.0     A  0.180759  0.059879
1  C_2    1  12.011     0.0     A  0.367383  0.583853
2  C_1    1  12.011     0.0     A  0.323320  0.265432
3   H_    1   1.008     0.0     A  0.268181  0.169900
4  N_R    1  14.007     0.0     A  0.320617  0.720316, 'bonds':   atom1 atom2 func  length  force_const
0   Zn3   N_R    1  0.1883    491035.36
1   C_2   C_2    1  0.1444    343088.00
2   C_2   C_1    1  0.1490    334720.00
3   C_1   N_R    1  0.1335    410032.00
4   C_1    H_    1  0.1080    284512.00
5   N_R   N_R    1  0.1320    418400.00, 'angles':   atom1 atom2 func atom3   angle  force_const
0   Zn3   N_R  C_1     1  132.60     1615.840
1   N_R   Zn3  N_R     1  109.47      301.050
2   Zn3   N_R  N_R     1  121.90     1615.840
3   C_1   C_2  C_2     1  112.70      488.273
4   C_2   C_1   H_     1  130.70      292.880
5   C_2   C_1  N_R     1  106.60      585.760
6   C_1   C_2  C_1     1  120.00      527.184


In [23]:
#Setting up loop for to create initial GMX reference data
for a, l1 in enumerate(frames):
    frame_number = f'F{indx[a]}'
    D_number = new_D

    # Create the directory D*/F*D* if it doesn't exist
    folder_path = os.path.join(GMX_ref_path, D_number, f'{frame_number}{D_number}')
    os.makedirs(folder_path, exist_ok=True)

        #.gro file
    FFOPTI_tools.write_gro(Structure_name, l1, f'{folder_path}', Structure)
    param_list = param[f'{D_number}']
        #MOF.itp file
    subprocess.run(["cp", f"{GMX_ref_path}/MOF.itp", f"{folder_path}"],check=True)
        #Force_field.itp file
    FFOPTI_tools.ref_ffitp_remake(f'{folder_path}', param_list)
        #topol.top
    Structure.write_top_file(print_directory=f'{folder_path}')
        #SP.mdp
    initial_opti_info.write_mdp_file(print_directory=f'{folder_path}')
    print(f'files created for {D_number}{frame_number}')
print('initial files created')


grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/F1157D3/SP.mdp written
files created for D3F1157
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/F2869D3/SP.mdp written
files created for D3F2869
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/F1158D3/SP.mdp written
files created for D3F1158
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/F2785D3/SP.mdp written
files created for D3F2785
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/F2480D3/SP.mdp written
files created for D3F2480
grofile created
force_field.itp file printed
topol.top file printed
/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/F1731D3/SP.

In [24]:
new_D

'D3'

In [26]:
tools = FFOPTI_tools()
tools.job_script_mpi(Structure_name, GMX_ref_path, new_D, indx)
print('submitted')
subprocess.run(["bash", "-l", f'{GMX_ref_path}/{new_D}/run_all'],check=True)

/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/run_all written
submitted


Unloading compilers/gnu/10.2.0
  Unloading dependent: cp2k/8.2/ompi/gnu-10.2.0 mpi/openmpi/4.0.5/gnu-10.2.0
Unloading compilers/intel/2018/update3
  Unloading dependent: gromacs/2019.3/plumed/intel-2018 plumed/2.5.2/intel-2018

Unloading gcc-libs/10.2.0
  Unloading dependent: flex/2.5.39 libmatheval/1.1.11
    openblas/0.3.13-openmp/gnu-10.2.0 ucx/1.9.0/gnu-10.2.0
    binutils/2.36.1/gnu-10.2.0
Unloading compilers/gnu/10.2.0
  Unloading dependent: cp2k/8.2/ompi/gnu-10.2.0 mpi/openmpi/4.0.5/gnu-10.2.0
                      :-) GROMACS - gmx grompp, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Kark

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


starting mdrun 'MOF'
0 steps,      0.0 ps.

Writing final coordinates.

NOTE: 16 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:      133.380        3.708     3597.1
                 (ns/day)    (hour/ns)
Performance:        0.023     1029.983

GROMACS reminds you: "Oh My God ! It's the Funky Shit" (Beastie Boys)

                       :-) GROMACS - gmx dump, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Karkoulis
    Peter Kasson        Jiri Kraus      Carsten Kut

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


...

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


starting mdrun 'MOF'
0 steps,      0.0 ps.

Writing final coordinates.

NOTE: 25 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:       72.543        2.038     3559.7
                 (ns/day)    (hour/ns)
Performance:        0.042      566.079

GROMACS reminds you: "Bum Stikkie Di Bum Stikkie Di Bum Stikkie Di Bum" (R. Slijngaard)

                       :-) GROMACS - gmx dump, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrgang  
  Aleksei Iupinov   Christoph Junghans     Joe Jordan     Dimitrios Karkoulis
    Peter Kasson        Jiri Krau

Analysing residue names:
There are:     1      Other residues
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 298 K
Calculated rlist for 1x1 atom pair-list as 1.200 nm, buffer size 0.000 nm
Set rlist, assuming 4x4 atom pair-list, to 1.200 nm, buffer size 0.000 nm
Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 24x24x25, spacing 0.110 0.110 0.117
This run will generate roughly 0 Mb of data


starting mdrun 'MOF'
0 steps,      0.0 ps.

Writing final coordinates.

NOTE: 25 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:       72.006        2.000     3599.4
                 (ns/day)    (hour/ns)
Performance:        0.043      555.689

GROMACS reminds you: "This work contains many things which are new and interesting. Unfortunately, everything that is new is not interesting, and everything which is interesting, is not new." (Lev Landau)

                       :-) GROMACS - gmx dump, 2019.3 (-:

                            GROMACS is written by:
     Emile Apol      Rossen Apostolov      Paul Bauer     Herman J.C. Berendsen
    Par Bjelkmar      Christian Blau   Viacheslav Bolnykh     Kevin Boyd    
 Aldert van Buuren   Rudi van Drunen     Anton Feenstra       Alan Gray     
  Gerrit Groenhof     Anca Hamuraru    Vincent Hindriksen  M. Eric Irrg

CompletedProcess(args=['bash', '-l', '/home/uccaset/Scratch/FFOPTI/data/GF334_v2/classical/D3/run_all'], returncode=0)

## Processing output and adding result to pkl

In [27]:
bob = param

In [29]:
print(param)

{'D1': {'lj':   atom func    mass  charge ptype     sigma  epsilon
0  Zn3    1  65.380     0.0     A  0.246155   0.5188
1  C_2    1  12.011     0.0     A  0.343085   0.4393
2  C_1    1  12.011     0.0     A  0.343085   0.4393
3   H_    1   1.008     0.0     A  0.257113   0.1841
4  N_R    1  14.007     0.0     A  0.326069   0.2887, 'bonds':   atom1 atom2 func  length  force_const
0   Zn3   N_R    1  0.1883    491035.36
1   C_2   C_2    1  0.1444    343088.00
2   C_2   C_1    1  0.1490    334720.00
3   C_1   N_R    1  0.1335    410032.00
4   C_1    H_    1  0.1080    284512.00
5   N_R   N_R    1  0.1320    418400.00, 'angles':   atom1 atom2 func atom3   angle  force_const
0   Zn3   N_R  C_1     1  132.60     1615.840
1   N_R   Zn3  N_R     1  109.47      301.050
2   Zn3   N_R  N_R     1  121.90     1615.840
3   C_1   C_2  C_2     1  112.70      488.273
4   C_2   C_1   H_     1  130.70      292.880
5   C_2   C_1  N_R     1  106.60      585.760
6   C_1   C_2  C_1     1  120.00      527.184

In [31]:
#calculating the loss for the new output
d_folder = f'{GMX_ref_path}/{new_D}'
single_calc_tools = single_loss_calc(Structure_name,Structure, DFT_ref_path,d_folder)

In [32]:
new_loss = single_calc_tools.average_type_loss_per_D()

Extracted forces from 100 frames.
Extracted forces from 100 frames.


In [33]:
new_loss

,3
Zn3,49.140344
C_2,0.735957
C_1,5.486122
H_,0.053464
N_R,36.817142


In [42]:
average_loss = average_loss.join(new_loss)

In [ ]:
average_loss.to_pickle(f"{GMX_ref_path}/ffopti_backup/param_type_loss.pkl")

In [43]:
len(average_loss.columns)

3